<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/RNNs_By_Hand_basic.ipynb)

# RNNs by Hand
--------------------------------
**Dr. Dave Wanik - University of Connecticut**

The companion notebook, **RNN Samples and the Hidden State**, answered *what a recurrent layer does*: it turns one
sample into a **sequence of hidden states**. This notebook answers the other question - ***how big is the model?***

Counting trainable parameters by hand, and naming each layer's output shape, is how you prove to yourself that you
know what the layer is doing. It is also exactly what **Assignment 5** asks for.

**The running example, in both notebooks:** three stocks - **AAPL, MSFT, TSLA** daily % changes - used to
predict **Netflix's price tomorrow**. Made-up numbers, so the arithmetic stays readable. A look-back of 5 means each
sample is the last 5 days, and one sample is a `(5, 3)` block: 5 steps, 3 features per step.

**How every example works below:** build the model → read `model.summary()` → do the arithmetic in $\LaTeX$ →
run `check()`, which compares your hand count to Keras **layer by layer** and raises if they disagree. Nothing in this
notebook is a number I typed in a comment; every number is checked against Keras when the notebook runs.

### Readings
* Counting parameters by hand: https://towardsdatascience.com/counting-no-of-parameters-in-deep-learning-models-by-hand-8f1716241889
* Animated RNN, LSTM and GRU: https://towardsdatascience.com/animated-rnn-lstm-and-gru-ef124d06cf45

### The five words we'll use all module

Everything in Module 4 is one of these five things. The GIFs use the same colors.

| Word | Symbol / code | In the GIFs | The one thing to remember |
| :-- | :-- | :-- | :-- |
| **look-back** | `n_steps` | the green window sliding down the series | how many time steps are in ONE sample |
| **features** | $i$, `n_features` | the **green dots**, one row per step | how many columns you feed in at each step |
| **hidden units** | $h$, the number in `SimpleRNN(h)` | the **red dots** | how wide the layer's memory is - *you* pick this |
| **hidden state** | $h_t$ | one set of red dots | the layer's memory **after** reading step $t$ |
| **sequence of hidden states** | $h_1 \dots h_T$ | all the red dots, left to right | what the layer actually produces: one hidden state per step |

A recurrent layer reads a sample one step at a time and emits a **sequence of hidden states**, one per time step.
`return_sequences=False` (the default) hands back only the **last** one, $h_T$; `return_sequences=True` hands back
**all $T$ of them**, which is a brand-new sequence another recurrent layer can read.

> **The look-back is not a feature.** It never appears in a parameter count. A layer that reads 5 steps and a layer
> that reads 30,000,000 steps have exactly the same weights - the same small network, reused at every step.

### One picture to keep on screen while you count

![a SimpleRNN(2) reading 5 days, then Dense](https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module4/img/m4_simplernn_lookback5_to_dense.gif)

The cell animations further down (the ones from the *Animated RNN, LSTM and GRU* post) are great for seeing **inside**
one cell, but they spin forever and never stop on the result. This one stops: it reads all 5 days, leaves the whole
**sequence of hidden states** on screen, and then sends **only the last one** into `Dense`.

Keep it in view. Every count below is just: **how many weights does that one cell hold?** - because that same cell is
reused at every step of the look-back.

In [1]:
import numpy as np
import pandas as pd
import keras
from keras import Input
from keras.models import Sequential
from keras.layers import SimpleRNN, LSTM, GRU, Dense
keras.utils.set_random_seed(5509)   # reproducibility: same numbers every run

## The only two formulas in this notebook

**Every recurrent layer** - SimpleRNN, GRU, LSTM - is the same formula. The only thing that changes is $g$, how many
little feed-forward networks are packed inside one cell:

$$
\text{recurrent params} \;=\; g\,\big[\;\underbrace{h\,(h+i)}_{\text{weights}} \;+\; \underbrace{h}_{\text{biases}}\;\big]
\qquad\quad g = \begin{cases} 1 & \textbf{SimpleRNN} \\ 3 & \textbf{GRU} \\ 4 & \textbf{LSTM} \end{cases}
$$

**Every dense layer** is the plain one you already know:

$$
\text{dense params} \;=\; \underbrace{n_{in}\,n_{out}}_{\text{weights}} \;+\; \underbrace{n_{out}}_{\text{biases}}
$$

Why $h(h+i)$: each of the $h$ units looks at the $i$ features arriving at this step **and** at the $h$ hidden values
handed over from the step before. So each unit has $h + i$ incoming weights, plus one bias. Then multiply by $g$,
because a GRU cell runs that little network 3 times and an LSTM cell runs it 4 times.

**Stacking rule:** a layer's $i$ is *the previous layer's $h$*. The features only matter for the first layer.

In [2]:
def rnn_params(g, h, i, reset_after=False):
    """Trainable parameters in one recurrent layer.
    g = little networks in a cell (SimpleRNN 1, GRU 3, LSTM 4)
    h = hidden units (the red dots)      i = features arriving each step (the green dots)
    reset_after=True is Keras' GRU default: a SECOND bias vector per gate, so 2h instead of h."""
    return g * (h * (h + i) + (2 * h if reset_after else h))

def dense_params(n_in, n_out):
    return n_in * n_out + n_out

def check(model, **by_hand):
    """Compare a hand count to Keras, layer by layer, and raise if any line disagrees.
    Pass one keyword per layer, in order: check(model, simple_rnn=12, dense=3)"""
    rows = []
    for (label, hand), layer in zip(by_hand.items(), model.layers):
        rows.append({"layer": layer.name, "by hand": hand, "Keras": layer.count_params(),
                     "output shape": str(layer.output.shape)})
    rows.append({"layer": "TOTAL", "by hand": sum(by_hand.values()), "Keras": model.count_params(), "output shape": ""})
    out = pd.DataFrame(rows)
    out["match"] = out["by hand"] == out["Keras"]
    assert out["match"].all(), "the hand count disagrees with Keras:\n" + out.to_string()
    return out

def anatomy(layer):
    """Open the layer up: the weight tensors Keras actually built, and how many numbers are in each one."""
    rows = [{"weight": w.name, "shape": tuple(w.shape), "numbers": int(np.prod(w.shape))} for w in layer.weights]
    rows.append({"weight": "TOTAL", "shape": "", "numbers": layer.count_params()})
    return pd.DataFrame(rows)

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 2 — The vanilla RNN, and why the look-back never costs you a parameter
- Pick up exactly where the hidden-state notebook stopped: the same 5 days x 3 stocks, the same SimpleRNN(2).
- The cell is a small dense net with tanh: each of the 2 units sees 3 green dots + 2 hidden values handed over = 5 inputs, +1 bias. 2 x (5+1) = 12.
- Plus the Dense(1) head on the 2 red dots = 3. Total 15 - point at summary() on screen.
- Then the punchline: change the look-back from 5 to 50 to 30,000,000 and the count NEVER moves. Same small network, reused at every step.
- Say the general formula once, G x [H(H+I) + H], and tell them G is the only thing that changes for GRU and LSTM.
-->


## Example 1 · the model from the hidden-state notebook

Same sample: **5 days × 3 stocks**, predicting Netflix's price. `SimpleRNN(2)` means **two red dots**, and those two
numbers are all the `Dense(1)` head ever sees.

In [3]:
model = Sequential([Input((5, 3)),        # (look-back, features) - one sample is a 5 x 3 block
                    SimpleRNN(2),        # 2 hidden units = the 2 red dots
                    Dense(1)])           # linear output: Netflix's price tomorrow
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 2)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15 (60.00 B)

 Trainable params: 15 (60.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 1, written out
**`SimpleRNN(2)` on 3 features** · `input_shape=(5, 3)`

$$
\begin{aligned}
\textbf{SimpleRNN}(2) & = g\,[\,h(h+i) + h\,] = 1\,[\,2(2+3) + 2\,] = 1\,[\,10 + 2\,] = \mathbf{12} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 2\cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 12 + 3 = \mathbf{15}
\end{aligned}
$$

$g = 1$ (one little network), $h = 2$ (the red dots), $i = 3$ (the green dots). The look-back, 5, is **not** in there.

In [4]:
check(model,
      simple_rnn=rnn_params(g=1, h=2, i=3),
      dense=dense_params(n_in=2, n_out=1))

,layer,by hand,Keras,output shape,match
0,simple_rnn,12,12,"(None, 2)",True
1,dense,3,3,"(None, 1)",True
2,TOTAL,15,15,,True


### Where those 12 numbers actually live

Don't take the formula on faith - open the layer up. Keras built **three** tensors, and they are exactly the three
pieces of $h(h+i) + h$:

| tensor | what it is | in the GIF |
| :-- | :-- | :-- |
| `kernel` $(i, h)$ | today's features → the hidden units | green dots → red dots |
| `recurrent_kernel` $(h, h)$ | the previous hidden state → the hidden units | red dots → red dots |
| `bias` $(h,)$ | one per hidden unit | — |

$3\times2 + 2\times2 + 2 = 6 + 4 + 2 = 12.$

In [5]:
anatomy(model.layers[0])      # the SimpleRNN(2): kernel, recurrent_kernel, bias

,weight,shape,numbers
0,kernel,"(3, 2)",6
1,recurrent_kernel,"(2, 2)",4
2,bias,"(2,)",2
3,TOTAL,,12


### Proof that the look-back is free

Three models, three wildly different look-backs, **one parameter count**. This is the single most common exam mistake:
putting `n_steps` into the formula.

In [6]:
for n_steps in [5, 50, 30_000_000]:
    m = Sequential([Input((n_steps, 3)), SimpleRNN(2), Dense(1)])
    print(f"look-back {n_steps:>10,}  ->  {m.count_params()} trainable parameters")

look-back          5  ->  15 trainable parameters
look-back         50  ->  15 trainable parameters
look-back 30,000,000  ->  15 trainable parameters


## Example 2 · stacking: the model from the GIF

This is the stacked model at the end of the hidden-state notebook. The first layer keeps **all five** hidden states
(`return_sequences=True`), which is a new `(5, 2)` sequence; the second layer reads *that* the same way the first one
read the stocks. Only the last recurrent layer drops `return_sequences`, so its final hidden state can go to `Dense`.

In [7]:
stacked = Sequential([Input((5, 3)),
                      SimpleRNN(2, return_sequences=True),   # hands on all 5 hidden states -> shape (5, 2)
                      SimpleRNN(2),                          # reads that sequence, returns only the last one
                      Dense(1)])
stacked.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 5, 2)           │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25 (100.00 B)

 Trainable params: 25 (100.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 2, written out
**stacked `SimpleRNN(2)` → `SimpleRNN(2)`** · `input_shape=(5, 3)`

$$
\begin{aligned}
\textbf{SimpleRNN}(2)\text{ on 3 stocks} & = 1\,[\,2(2+3) + 2\,] = \mathbf{12} \\
\textbf{SimpleRNN}(2)\text{ on 2 hidden units} & = 1\,[\,2(2+2) + 2\,] = \mathbf{10} \\
\textbf{Dense}(1) & = 2 \cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 12 + 10 + 3 = \mathbf{25}
\end{aligned}
$$

The second layer's $i$ is **2**, not 3: its inputs are the first layer's red dots. **Every layer's $i$ is the previous
layer's $h$.**

In [8]:
check(stacked,
      simple_rnn_1=rnn_params(g=1, h=2, i=3),
      simple_rnn_2=rnn_params(g=1, h=2, i=2),    # i = the first layer's 2 hidden units
      dense=dense_params(n_in=2, n_out=1))

,layer,by hand,Keras,output shape,match
0,simple_rnn_4,12,12,"(None, 5, 2)",True
1,simple_rnn_5,10,10,"(None, 2)",True
2,dense_4,3,3,"(None, 1)",True
3,TOTAL,25,25,,True


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 3 — Same formula, bigger numbers
- 30 stocks into 25 hidden units: (30+25) x 25 + 25 = 1,400, plus a 26-parameter dense head = 1,426.
- Nothing new - just show that h and i are the only two dials.
- Output shape is ALWAYS (None, h) when return_sequences is off, and (None, look-back, h) when it's on.
- Assignment 5 grades exactly this: match every line of summary().
-->


## Example 3 · same formula, bigger numbers

Thirty stocks now, and a much wider memory: `SimpleRNN(25)`.

In [9]:
model = Sequential([Input((50, 30)), SimpleRNN(25), Dense(1)])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_6 (SimpleRNN)        │ (None, 25)             │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,426 (5.57 KB)

 Trainable params: 1,426 (5.57 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 3, written out
**`SimpleRNN(25)` on 30 features** · `input_shape=(50, 30)`

$$
\begin{aligned}
\textbf{SimpleRNN}(25) & = 1\,[\,25(25+30) + 25\,] = 1\,[\,1{,}375 + 25\,] = \mathbf{1{,}400} \\
\textbf{Dense}(1) & = 25\cdot 1 + 1 = \mathbf{26} \\
\textbf{Total} & = 1{,}400 + 26 = \mathbf{1{,}426}
\end{aligned}
$$

In [10]:
check(model,
      simple_rnn=rnn_params(g=1, h=25, i=30),
      dense=dense_params(n_in=25, n_out=1))

,layer,by hand,Keras,output shape,match
0,simple_rnn_6,1400,1400,"(None, 25)",True
1,dense_5,26,26,"(None, 1)",True
2,TOTAL,1426,1426,,True


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 4 — LSTM by hand: four networks and a cell state
- G=4: four little networks, plus a CELL state (long memory) running alongside the hidden state (recent memory) - that's the vanishing-gradient fix.
- Identical arithmetic to Example 1, times four: 12 -> 48. Show the two side by side.
- The cell state does NOT add parameters - it's carried, not learned. Students ask this every year.
- Why LSTMs are slow: every time step spins all four networks.
- Forget / input / output gates decide what to keep - but at heart it's four small nets fit at once.
-->


## Example 4 · one LSTM layer

An LSTM cell holds **four** little networks instead of one, so $g = 4$. It also carries a **cell state** alongside the
hidden state - long memory beside recent memory - but that state is *carried*, not *learned*: it adds **no**
parameters.

![LSTM cell](https://miro.medium.com/max/2250/1*goJVQs-p9kgLODFNyhl9zA.gif)

*(This one loops forever and never shows you the answer — it is a picture of the **inside of one cell**. For what the layer produces across the whole look-back, scroll back to the GIF at the top.)*

In [11]:
model = Sequential([Input((5, 3)), LSTM(2), Dense(1)])    # same sample as Example 1, LSTM instead of SimpleRNN
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 2)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51 (204.00 B)

 Trainable params: 51 (204.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 4, written out
**`LSTM(2)` on 3 features** · `input_shape=(5, 3)`

$$
\begin{aligned}
\textbf{LSTM}(2) & = g\,[\,h(h+i) + h\,] = 4\,[\,2(2+3) + 2\,] = 4\,[\,10 + 2\,] = \mathbf{48} \\
\textbf{Dense}(1) & = 2\cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 48 + 3 = \mathbf{51}
\end{aligned}
$$

Exactly Example 1's bracket, $\times 4$: $12 \to 48$.

In [12]:
check(model,
      lstm=rnn_params(g=4, h=2, i=3),
      dense=dense_params(n_in=2, n_out=1))

,layer,by hand,Keras,output shape,match
0,lstm,48,48,"(None, 2)",True
1,dense_6,3,3,"(None, 1)",True
2,TOTAL,51,51,,True


### You can *see* the four networks

Same three tensors as the SimpleRNN, but every one of them is **four times wider**: Keras packs the four gates
side by side into one matrix. Look at the shapes - `(3, 8)` instead of `(3, 2)`, because $8 = 4 \text{ gates} \times
2 \text{ units}$. That column count **is** $g$.

In [13]:
print("SimpleRNN(2) on the same sample:"); display(anatomy(Sequential([Input((5, 3)), SimpleRNN(2)]).layers[0]))
print("\nLSTM(2) - every tensor is 4x wider:");    display(anatomy(model.layers[0]))

SimpleRNN(2) on the same sample:


,weight,shape,numbers
0,kernel,"(3, 2)",6
1,recurrent_kernel,"(2, 2)",4
2,bias,"(2,)",2
3,TOTAL,,12



LSTM(2) - every tensor is 4x wider:


,weight,shape,numbers
0,kernel,"(3, 8)",24
1,recurrent_kernel,"(2, 8)",16
2,bias,"(8,)",8
3,TOTAL,,48


## Example 5 · a wider LSTM

Five stocks, four red dots, and a look-back of 30.

In [14]:
model = Sequential([Input((30, 5)), LSTM(4), Dense(1)])
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 4)              │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165 (660.00 B)

 Trainable params: 165 (660.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 5, written out
**`LSTM(4)` on 5 features** · `input_shape=(30, 5)`

$$
\begin{aligned}
\textbf{LSTM}(4) & = 4\,[\,4(4+5) + 4\,] = 4\,[\,36 + 4\,] = \mathbf{160} \\
\textbf{Dense}(1) & = 4\cdot 1 + 1 = \mathbf{5} \\
\textbf{Total} & = 160 + 5 = \mathbf{165}
\end{aligned}
$$

In [15]:
check(model,
      lstm=rnn_params(g=4, h=4, i=5),
      dense=dense_params(n_in=4, n_out=1))

,layer,by hand,Keras,output shape,match
0,lstm_1,160,160,"(None, 4)",True
1,dense_7,5,5,"(None, 1)",True
2,TOTAL,165,165,,True


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 5 — GRU by hand, and the reset_after trap
- G=3: reset and update gates, same bracket x3 -> 36. Two gates, three networks - candidate, reset, update.
- THE TRAP: Keras' GRU defaults to reset_after=True, a SECOND bias vector per gate, so h becomes 2h. Set reset_after=False and the textbook formula matches. Say it out loud or summary() contradicts you on screen.
- Side by side on the same 5x3 sample, one hidden unit count: SimpleRNN 12, GRU 36, LSTM 48. That is the whole of g.
- STOP HERE. Everything below this point is stacking, and that is M4.2 (video 13, the week before Assignment 5).
- Tell them it's there on purpose: read ahead, try the arithmetic, the check() call is the answer key.
-->


## Example 6 · one GRU layer (and the `reset_after` trap)

A GRU cell holds **three** little networks, so $g = 3$.

⚠️ **Read this before you argue with `summary()`.** Keras' `GRU` defaults to **`reset_after=True`**, which gives
every gate a *second* bias vector - so the bias term is $2h$, not $h$, and the count comes out higher than the
textbook formula. Set `reset_after=False` and the one formula works. Example 8 below uses the Keras default on
purpose, so you have seen both.

![GRU cell](https://miro.medium.com/max/2214/1*lNNJOWnMjxLzdUnUQqwKcw.gif)

*(Same caveat: inside one cell, looping. Three little networks feeding one hidden state.)*

In [16]:
model = Sequential([Input((5, 3)),
                    GRU(2, reset_after=False),    # False -> one bias vector per gate, so the textbook formula matches
                    Dense(1)])
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 2)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39 (156.00 B)

 Trainable params: 39 (156.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 6, written out
**`GRU(2)` on 3 features, `reset_after=False`** · `input_shape=(5, 3)`

$$
\begin{aligned}
\textbf{GRU}(2) & = g\,[\,h(h+i) + h\,] = 3\,[\,2(2+3) + 2\,] = 3\,[\,10 + 2\,] = \mathbf{36} \\
\textbf{Dense}(1) & = 2\cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 36 + 3 = \mathbf{39}
\end{aligned}
$$

Same bracket as Example 1, $\times 3$. Compare the three cells on the same sample: **12 · 36 · 48**.

In [17]:
check(model,
      gru=rnn_params(g=3, h=2, i=3),
      dense=dense_params(n_in=2, n_out=1))

,layer,by hand,Keras,output shape,match
0,gru,36,36,"(None, 2)",True
1,dense_8,3,3,"(None, 1)",True
2,TOTAL,39,39,,True


### Both GRU biases, side by side

Here is the whole trap in one screen. The `kernel` and `recurrent_kernel` are identical - only the **bias tensor**
changes, from `(6,)` to `(2, 6)`. That extra row is the second bias vector, and it is worth $2h - h = h$ extra
parameters per gate: $3 \times 2 = 6$ more in total.

In [18]:
for ra in [False, True]:
    m = Sequential([Input((5, 3)), GRU(2, reset_after=ra)])
    print(f"=== reset_after={ra}  ->  Keras {m.count_params()}, by hand {rnn_params(3, 2, 3, reset_after=ra)}"
          f"   (bias term = {'2h' if ra else 'h'})")
    display(anatomy(m.layers[0]))

=== reset_after=False  ->  Keras 36, by hand 36   (bias term = h)


,weight,shape,numbers
0,kernel,"(3, 6)",18
1,recurrent_kernel,"(2, 6)",12
2,bias,"(6,)",6
3,TOTAL,,36


=== reset_after=True  ->  Keras 42, by hand 42   (bias term = 2h)


,weight,shape,numbers
0,kernel,"(3, 6)",18
1,recurrent_kernel,"(2, 6)",12
2,bias,"(2, 6)",12
3,TOTAL,,42


## Example 7 · a wider GRU

Five stocks, four red dots - and a deliberately absurd look-back, to make the point one last time.

In [19]:
model = Sequential([Input((30_000_000, 5)),          # thirty million time steps!
                    GRU(4, reset_after=False),
                    Dense(1)])
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_3 (GRU)                     │ (None, 4)              │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125 (500.00 B)

 Trainable params: 125 (500.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 7, written out
**`GRU(4)` on 5 features, `reset_after=False`** · `input_shape=(30{,}000{,}000, 5)`

$$
\begin{aligned}
\textbf{GRU}(4) & = 3\,[\,4(4+5) + 4\,] = 3\,[\,36 + 4\,] = \mathbf{120} \\
\textbf{Dense}(1) & = 4\cdot 1 + 1 = \mathbf{5} \\
\textbf{Total} & = 120 + 5 = \mathbf{125}
\end{aligned}
$$

Thirty million time steps, 125 parameters. **The count depends on $h$ and $i$ only.**

In [20]:
check(model,
      gru=rnn_params(g=3, h=4, i=5),
      dense=dense_params(n_in=4, n_out=1))

,layer,by hand,Keras,output shape,match
0,gru_3,120,120,"(None, 4)",True
1,dense_9,5,5,"(None, 1)",True
2,TOTAL,125,125,,True


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 13 — Stacking and mixing cells: the i-chain, and the monsters
- Open on the by-hand notebook, Examples 8-11 - the section students were told to read ahead in M4.1. This is where it gets taught.
- ONE rule does all of it: each layer's i is the previous layer's h. Walk the chain down Example 8 on screen: 5 -> 4 -> 2 -> 25.
- Every recurrent layer except the last needs return_sequences=True. Forget it and you hand a 2-D tensor to a layer that wants 3-D - show the error once, it saves a hundred emails.
- Mixing cell types is a one-word swap in Keras: SimpleRNN -> LSTM -> GRU, everything else identical. Costs differ by g, nothing else does.
- More layers is NOT automatically better - it's a hyperparameter, and the small-model advice from Week 4 still applies.
- Then the two monsters. They're exercises: pause, let them count, then run check(). After this, come back to Advanced_RNN_Theory for Conv1D, recurrent dropout and bidirectional.
- Assignment 5 (RNN Math, due Nov 6) is exactly this - say so.
-->


# Stacking, mixing and matching

> ⚠️ **This section is Module 4.2 material.** Everything above is one recurrent layer, which is all you need for
> the univariate notebooks. Everything below is what happens when you put layers on top of each other - we teach it
> the week before **Assignment 5**, alongside Conv1D, recurrent dropout and bidirectional in `Advanced_RNN_Theory`.
> Read ahead if you like: the `check()` calls are the answer key.

One rule handles all of it: **each layer's $i$ is the previous layer's $h$.** Every recurrent layer except the last
one needs `return_sequences=True`, so it hands on its whole sequence of hidden states instead of just the final one.

The rest of this notebook is practice. Try each one before you read the answer - and note that the `check()` call
*is* the answer key: if your arithmetic is right, it runs; if it's wrong, it raises.

### Example 8 · two GRU layers into a SimpleRNN

These GRUs use the **Keras default** `reset_after=True`, so their bias term is $2h$. This is the version you'll meet
in the wild.

In [21]:
model = Sequential([Input((30, 5)),
                    GRU(4, return_sequences=True),          # -> (None, 30, 4): all 30 hidden states
                    GRU(2, return_sequences=True),          # -> (None, 30, 2)
                    SimpleRNN(25),                          # -> (None, 25): last hidden state only
                    Dense(1)])
model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 30, 4)          │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 30, 2)          │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_8 (SimpleRNN)        │ (None, 25)             │           700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 906 (3.54 KB)

 Trainable params: 906 (3.54 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Example 8, written out
**GRU → GRU → SimpleRNN, Keras-default GRUs** · `input_shape=(30, 5)`

$$
\begin{aligned}
\textbf{GRU}(4) & = 3\,[\,4(4+5) + 2\cdot4\,] = 3\,[\,36 + 8\,] = \mathbf{132} \\
\textbf{GRU}(2) & = 3\,[\,2(2+4) + 2\cdot2\,] = 3\,[\,12 + 4\,] = \mathbf{48} \\
\textbf{SimpleRNN}(25) & = 1\,[\,25(25+2) + 25\,] = 1\,[\,675 + 25\,] = \mathbf{700} \\
\textbf{Dense}(1) & = 25\cdot 1 + 1 = \mathbf{26} \\
\textbf{Total} & = \mathbf{906}
\end{aligned}
$$

Follow the $i$ down the stack: **5 → 4 → 2 → 25**. Each one is the layer above it.

In [22]:
check(model,
      gru_1=rnn_params(g=3, h=4, i=5,  reset_after=True),    # Keras default: 2h bias
      gru_2=rnn_params(g=3, h=2, i=4,  reset_after=True),    # i = gru_1's 4 units
      simple_rnn=rnn_params(g=1, h=25, i=2),                 # i = gru_2's 2 units
      dense=dense_params(n_in=25, n_out=1))

,layer,by hand,Keras,output shape,match
0,gru_4,132,132,"(None, 30, 4)",True
1,gru_5,48,48,"(None, 30, 2)",True
2,simple_rnn_8,700,700,"(None, 25)",True
3,dense_10,26,26,"(None, 1)",True
4,TOTAL,906,906,,True


### Example 9 · a SimpleRNN into an LSTM
*Try it yourself before running the answer.*

In [23]:
model = Sequential([Input((15, 30)),
                    SimpleRNN(20, return_sequences=True),
                    LSTM(4),                                 # no return_sequences: last hidden state only
                    Dense(1)])
model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_9 (SimpleRNN)        │ (None, 15, 20)         │         1,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 4)              │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,425 (5.57 KB)

 Trainable params: 1,425 (5.57 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Answer — Example 9, written out
**`SimpleRNN(20)` → `LSTM(4)`** · `input_shape=(15, 30)`

$$
\begin{aligned}
\textbf{SimpleRNN}(20) & = 1\,[\,20(20+30) + 20\,] = \mathbf{1{,}020} \\
\textbf{LSTM}(4) & = 4\,[\,4(4+20) + 4\,] = 4\,[\,96 + 4\,] = \mathbf{400} \\
\textbf{Dense}(1) & = 4\cdot 1 + 1 = \mathbf{5} \\
\textbf{Total} & = \mathbf{1{,}425}
\end{aligned}
$$

The LSTM's $i$ is **20** - the SimpleRNN's hidden units - not the 30 stocks.

In [24]:
check(model,
      simple_rnn=rnn_params(g=1, h=20, i=30),
      lstm=rnn_params(g=4, h=4, i=20),
      dense=dense_params(n_in=4, n_out=1))

,layer,by hand,Keras,output shape,match
0,simple_rnn_9,1020,1020,"(None, 15, 20)",True
1,lstm_2,400,400,"(None, 4)",True
2,dense_11,5,5,"(None, 1)",True
3,TOTAL,1425,1425,,True


### Example 10 · Monster #1
*Three different cell types in a row, then two dense layers. Left as an exercise.*

In [25]:
model = Sequential([Input((30, 30)),
                    SimpleRNN(30, return_sequences=True),
                    GRU(30, return_sequences=True),          # Keras default reset_after=True
                    LSTM(30),
                    Dense(30, activation="relu"),
                    Dense(1)])
model.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_10 (SimpleRNN)       │ (None, 30, 30)         │         1,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 30, 30)         │         5,580 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 30)             │         7,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,691 (61.29 KB)

 Trainable params: 15,691 (61.29 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Answer — Example 10, written out
**Monster #1** · `input_shape=(30, 30)`

$$
\begin{aligned}
\textbf{SimpleRNN}(30) & = 1\,[\,30(30+30) + 30\,] = \mathbf{1{,}830} \\
\textbf{GRU}(30) & = 3\,[\,30(30+30) + 2\cdot30\,] = 3\,[\,1{,}800 + 60\,] = \mathbf{5{,}580} \\
\textbf{LSTM}(30) & = 4\,[\,30(30+30) + 30\,] = 4\,[\,1{,}800 + 30\,] = \mathbf{7{,}320} \\
\textbf{Dense}(30) & = 30\cdot 30 + 30 = \mathbf{930} \\
\textbf{Dense}(1) & = 30\cdot 1 + 1 = \mathbf{31} \\
\textbf{Total} & = \mathbf{15{,}691}
\end{aligned}
$$

In [26]:
check(model,
      simple_rnn=rnn_params(g=1, h=30, i=30),
      gru=rnn_params(g=3, h=30, i=30, reset_after=True),
      lstm=rnn_params(g=4, h=30, i=30),
      dense_1=dense_params(n_in=30, n_out=30),
      dense_2=dense_params(n_in=30, n_out=1))

,layer,by hand,Keras,output shape,match
0,simple_rnn_10,1830,1830,"(None, 30, 30)",True
1,gru_6,5580,5580,"(None, 30, 30)",True
2,lstm_3,7320,7320,"(None, 30)",True
3,dense_12,930,930,"(None, 30)",True
4,dense_13,31,31,"(None, 1)",True
5,TOTAL,15691,15691,,True


### Example 11 · Monster #2
*Six recurrent layers and five dense layers. Left as an exercise - go slowly and track $i$ down the stack.*

In [27]:
model = Sequential([Input((50, 40)),
                    SimpleRNN(30, return_sequences=True),
                    GRU(20, return_sequences=True),
                    GRU(25, return_sequences=True),
                    GRU(22, return_sequences=True),
                    GRU(21, return_sequences=True),
                    SimpleRNN(10),
                    Dense(50, activation="relu"),
                    Dense(50, activation="relu"),
                    Dense(50, activation="relu"),
                    Dense(50, activation="relu"),
                    Dense(1)])
model.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_11 (SimpleRNN)       │ (None, 50, 30)         │         2,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 50, 20)         │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 50, 25)         │         3,525 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_9 (GRU)                     │ (None, 50, 22)         │         3,234 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_10 (GRU)                    │ (None, 50, 21)         │         2,835 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_12 (SimpleRNN)       │ (None, 10)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 50)             │           550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,415 (91.46 KB)

 Trainable params: 23,415 (91.46 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Answer — Example 11, written out
**Monster #2** · `input_shape=(50, 40)`. The $i$ chain is **40 → 30 → 20 → 25 → 22 → 21 → 10**.

$$
\begin{aligned}
\textbf{SimpleRNN}(30) & = 1\,[\,30(30+40) + 30\,] = \mathbf{2{,}130} \\
\textbf{GRU}(20) & = 3\,[\,20(20+30) + 2\cdot20\,] = \mathbf{3{,}120} \\
\textbf{GRU}(25) & = 3\,[\,25(25+20) + 2\cdot25\,] = \mathbf{3{,}525} \\
\textbf{GRU}(22) & = 3\,[\,22(22+25) + 2\cdot22\,] = \mathbf{3{,}234} \\
\textbf{GRU}(21) & = 3\,[\,21(21+22) + 2\cdot21\,] = \mathbf{2{,}835} \\
\textbf{SimpleRNN}(10) & = 1\,[\,10(10+21) + 10\,] = \mathbf{320} \\
\textbf{Dense}(50) & = 10\cdot 50 + 50 = \mathbf{550} \\
\textbf{Dense}(50)\times 3 & = 50\cdot 50 + 50 = \mathbf{2{,}550}\text{ each} \\
\textbf{Dense}(1) & = 50\cdot 1 + 1 = \mathbf{51} \\
\textbf{Total} & = \mathbf{23{,}415}
\end{aligned}
$$

In [28]:
check(model,
      simple_rnn_1=rnn_params(g=1, h=30, i=40),
      gru_1=rnn_params(g=3, h=20, i=30, reset_after=True),
      gru_2=rnn_params(g=3, h=25, i=20, reset_after=True),
      gru_3=rnn_params(g=3, h=22, i=25, reset_after=True),
      gru_4=rnn_params(g=3, h=21, i=22, reset_after=True),
      simple_rnn_2=rnn_params(g=1, h=10, i=21),
      dense_1=dense_params(n_in=10, n_out=50),
      dense_2=dense_params(n_in=50, n_out=50),
      dense_3=dense_params(n_in=50, n_out=50),
      dense_4=dense_params(n_in=50, n_out=50),
      dense_5=dense_params(n_in=50, n_out=1))

,layer,by hand,Keras,output shape,match
0,simple_rnn_11,2130,2130,"(None, 50, 30)",True
1,gru_7,3120,3120,"(None, 50, 20)",True
2,gru_8,3525,3525,"(None, 50, 25)",True
3,gru_9,3234,3234,"(None, 50, 22)",True
4,gru_10,2835,2835,"(None, 50, 21)",True
5,simple_rnn_12,320,320,"(None, 10)",True
6,dense_14,550,550,"(None, 50)",True
7,dense_15,2550,2550,"(None, 50)",True
8,dense_16,2550,2550,"(None, 50)",True
9,dense_17,2550,2550,"(None, 50)",True


## The whole notebook on one card

| | SimpleRNN | GRU | LSTM |
| :-- | :-- | :-- | :-- |
| $g$ (little networks in a cell) | 1 | 3 | 4 |
| params | $1[h(h+i)+h]$ | $3[h(h+i)+h]$, or $3[h(h+i)+2h]$ with Keras' default `reset_after=True` | $4[h(h+i)+h]$ |
| on our 5×3 sample with $h=2$ | **12** | **36** | **48** |
| output, `return_sequences=False` | `(None, h)` | `(None, h)` | `(None, h)` |
| output, `return_sequences=True` | `(None, look-back, h)` | `(None, look-back, h)` | `(None, look-back, h)` |

Two things to carry out of **Module 4.1**:

1. **The look-back is never in the count.** $h$ and $i$ are the only dials.
2. **Keras' GRU defaults to `reset_after=True`** — the bias term is $2h$. Check which one the question wants.

And one more that arrives in **Module 4.2**, with the stacking section above:

3. **A stacked layer's $i$ is the previous layer's $h$.**

All three are on **Assignment 5** (RNN Math, due Nov 6).

Next up: put a real series through one of these. → **`Univariate_Temperature_RNN`**